In [2]:
print("Hello woeld")

Hello woeld


## Usando o pandas para analisar o arquivo
---
* Caminho: C:\Users\joao.victor\MSGÁS\GEOP - Documentos\2026\Manutenção - 2026

In [3]:
from calendar import month
from dbm import error
#instala as bibliotecas
#!pip install pandas
#!pip install openpyxl
!pip install dotenv

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)

   ---------------------------------------- 2/2 [dotenv]



In [4]:
#importa bibliotecas
import pandas as pd
from dotenv import load_dotenv
import os

ModuleNotFoundError: No module named 'pandas'

In [ ]:
load_dotenv()

camihho = os.getenv("FILE_BASE_DATA")
try:
    df = pd.read_excel(camihho, sheet_name='IFS_TASK_CLOCKING')
    print("Carregado dataframe")
except Exception as e:
    print('Erro ao ler o excel', e)

## Tratamento de dados
* Apenas colunas, Nome em PT-BR - Nome em ING-IFSfrescos
   * N° Tarefa - TASK_SEQ
   * Descr da Tarefa - TASK_DESCRIPTION
   * Categoria de Registro - CLOCKING_CATEGORY (Viagem ou serviço)
   * Tipo de Reg de Horas - CLOCKING_TYPE (Registro de saída ou de entrada)
   * Org de Manut - ORGANIZATION_ID
   * Horário de Início - START_TIME
   * Hora Parada - STOP_TIME
   * Horas Trab - WORK_HOURS
   * ID Recurso - EMPLOYEE_ID
* Limpa linhas que estão incompletas
* Modifica os dtypes das colunas
*

In [ ]:
essential_columns = [
  "TASK_SEQ",
  "TASK_DESCRIPTION",
  "CLOCKING_CATEGORY",
  "CLOCKING_TYPE",
  "START_TIME",
  "STOP_TIME",
  "WORK_HOURS",
  "ORGANIZATION_ID",
  "EMPLOYEE_ID"
]
df_clean = df[essential_columns]

In [ ]:
# mudando os nomes
df_clean.rename(columns={"TASK_SEQ" : "ID_tarefa",
  "TASK_DESCRIPTION": "Descricao",
  "CLOCKING_CATEGORY": "Tipo_temporal",
  "CLOCKING_TYPE": "Valida_registro",
  "START_TIME": "Tempo_inicio",
  "STOP_TIME": "Tempo_fim",
  "WORK_HOURS": "Horas_trabalhadas",
  "ORGANIZATION_ID": "ORG_manut",
  "EMPLOYEE_ID": "TOM"}, inplace=True)

In [ ]:
#Retirando dados que estão em execucao
df_clean.dropna(axis=0, how='any', inplace=True)

In [ ]:
#Retira serviços de terceiros ORG MANUT -> != MCGR, OCGR, TLG


In [ ]:
# Formata os tempo_inicio e tempo_fim
#       função de formatação
def formata_data(d):
    #Variaveis
    r = str(d) #Variavel de retorno
    month_number = {
        "jan": "01",
        "feb": "02",
        "mar": "03",
        "apr": "04",
        "may": "05",
        "jun": "06",
        "jul": "07",
        "aug": "08",
        "sep": "09",
        "oct": "10",
        "nov": "11",
        "dec": "12"
    } # Dicionario conversor de mes de nome para numero
    r = r.strip().lower().split() #tirando espaços e deoxando tudo minusculo
    day = r[1].replace(",", " ").strip()
    month = month_number[r[0]]
    year = r[2][:4]
    hour = pd.to_timedelta(r[3]) + pd.to_timedelta("12:00:00") if r[4] == "pm" else pd.to_timedelta(r[3]) #horas "brasileiras"
    hour = str(hour)[-8:] # Transform o dado em string
    #Formato antes: May 5, 2026, 2:37:30 PM
    #Formato que deve ficar 05/05/2026 02:37:30
    return day + "/" + month + "/" + year + " " + hour
#       aplica função de formatação
df_clean["Tempo_inicio"] = df_clean["Tempo_inicio"].apply(formata_data)
df_clean["Tempo_fim"] = df_clean["Tempo_fim"].apply(formata_data)

In [ ]:
# Formata colunas de tempo para deltatime
df_clean["Tempo_inicio"] = pd.to_datetime(df_clean["Tempo_inicio"], format="%d/%m/%Y %H:%M:%S")
df_clean["Tempo_fim"] = pd.to_datetime(df_clean["Tempo_fim"], format="%d/%m/%Y %H:%M:%S")

In [ ]:
filtro_temporal = df_clean["Tempo_inicio"].between(pd.to_datetime("04/01/2026"), pd.to_datetime("04/30/2026"))
df_periodo = df_clean[filtro_temporal]

## Analisando dados
* Novo df
|N° da tarefa|Decrição da tarefa|
|-------|----------|---------|


In [ ]:
df_periodo